# BÀI THỰC HÀNH CHƯƠNG 4 — NAMED ENTITY RECOGNITION & INFORMATION EXTRACTION

**Môn:** Xử lý ngôn ngữ tự nhiên  
**Chủ đề:** NER, BIO/BIOES, Gazetteer, Feature-based NER, CRF, đánh giá NER, Relation Extraction, Event/Template Filling

## Mục tiêu

Sau notebook này, sinh viên có thể:

1. Biểu diễn nhãn NER theo BIO.
2. Chuyển đổi giữa **token labels** và **entity spans**.
3. Kiểm tra một chuỗi BIO có hợp lệ hay không.
4. Xây dựng baseline NER bằng **gazetteer**.
5. Thiết kế các **feature bề mặt, ngôn ngữ và ngữ cảnh**.
6. Huấn luyện mô hình **Linear-chain CRF** cho NER.
7. Đánh giá NER bằng **token accuracy, exact span Precision/Recall/F1**.
8. Phân tích **strict matching** và **partial matching**.
9. Trích xuất **relation** và **event arguments** ở mức cơ bản.
10. Thực hiện **template filling** từ văn bản.

> Notebook được thiết kế theo hướng: **mẫu → TODO → đánh giá → phân tích lỗi**.

## Quy định bài làm

- Không thay đổi phần dữ liệu gốc trừ khi bài yêu cầu.
- Các ô có `TODO` là phần sinh viên phải hoàn thiện.
- Mỗi hàm cần có mô tả ngắn hoặc comment giải thích ý tưởng.
- Không chỉ báo cáo F1; phải có **error analysis**.
- Với các bài nâng cao, sinh viên có thể sử dụng thư viện ngoài nhưng phải ghi rõ.

### Thang điểm gợi ý

| Phần | Nội dung | Điểm |
|---|---|---:|
| 1 | BIO + span conversion + validator | 1.5 |
| 2 | Gazetteer NER | 1.5 |
| 3 | Feature engineering + CRF | 2.5 |
| 4 | Đánh giá + phân tích lỗi | 1.5 |
| 5 | Relation Extraction | 1.0 |
| 6 | Event/Template Filling | 1.5 |
| 7 | Bài nâng cao | +1.0 bonus |

# 0. Chuẩn bị môi trường

In [91]:
from collections import defaultdict, Counter
import re
import math
import json
from pprint import pprint

print("Environment ready.")

Environment ready.


Notebook sử dụng `sklearn-crfsuite` cho phần CRF. Nếu chưa cài, chạy ô sau.

In [92]:
# %pip install -q sklearn-crfsuite seqeval

In [93]:
import sklearn_crfsuite
import seqeval

# 1. Dữ liệu NER mẫu

Ta sử dụng một tập dữ liệu nhỏ để minh họa. Mỗi câu được biểu diễn thành `(tokens, BIO_labels)`.

Các loại thực thể:
- `PER`: Person
- `ORG`: Organization
- `LOC`: Location
- `DATE`: Date/Time
- `MONEY`: Money
- `POSITION`: Position

In [94]:
train_data = [
    (["Công_ty", "Sao_Bắc", "bổ_nhiệm", "bà", "Linh", "làm", "giám_đốc", "."],
     ["B-ORG", "I-ORG", "O", "O", "B-PER", "O", "B-POSITION", "O"]),
    (["FPT", "đặt", "trụ_sở", "tại", "Hà_Nội", "."],
     ["B-ORG", "O", "O", "O", "B-LOC", "O"]),
    (["Nguyễn_Văn_An", "làm_việc", "tại", "Viettel", "."],
     ["B-PER", "O", "O", "B-ORG", "O"]),
    (["VinFast", "mở", "nhà_máy", "mới", "ở", "Hải_Phòng", "."],
     ["B-ORG", "O", "O", "O", "O", "B-LOC", "O"]),
    (["Đại_học", "Quốc_gia", "Hà_Nội", "tổ_chức", "hội_thảo", "."],
     ["B-ORG", "I-ORG", "I-ORG", "O", "O", "O"]),
    (["Bà", "Mai", "đến", "Đà_Nẵng", "vào", "12/09/2026", "."],
     ["O", "B-PER", "O", "B-LOC", "O", "B-DATE", "O"]),
    (["Công_ty", "ABC", "tuyển", "kỹ_sư", "tại", "TP.HCM", "."],
     ["B-ORG", "I-ORG", "O", "O", "O", "B-LOC", "O"]),
    (["Apple", "mở", "văn_phòng", "tại", "Singapore", "."],
     ["B-ORG", "O", "O", "O", "B-LOC", "O"]),
]

test_data = [
    (["Công_ty", "Minh_Long", "bổ_nhiệm", "ông", "Nam", "làm", "phó_giám_đốc", "."],
     ["B-ORG", "I-ORG", "O", "O", "B-PER", "O", "B-POSITION", "O"]),
    (["Viettel", "mở", "chi_nhánh", "tại", "Đà_Nẵng", "."],
     ["B-ORG", "O", "O", "O", "B-LOC", "O"]),
    (["Lan", "gia_nhập", "FPT", "vào", "01/10/2026", "."],
     ["B-PER", "O", "B-ORG", "O", "B-DATE", "O"])
]

print("Train sentences:", len(train_data))
print("Test sentences :", len(test_data))

Train sentences: 8
Test sentences : 3


# 2. Bài 1 — BIO, span annotation và kiểm tra nhãn

## 2.1. Hàm mẫu: tách prefix và entity type

In [95]:
def split_bio_label(label):
    if label == "O":
        return "O", None
    prefix, entity_type = label.split("-", 1)
    return prefix, entity_type

for label in ["B-ORG", "I-PER", "O"]:
    print(label, "->", split_bio_label(label))

B-ORG -> ('B', 'ORG')
I-PER -> ('I', 'PER')
O -> ('O', None)


## 2.2. TODO 1 — Viết BIO validator

Một chuỗi BIO hợp lệ cần thỏa:
- `I-X` không được đứng đầu chuỗi.
- `I-X` chỉ được đi sau `B-X` hoặc `I-X`.
- `B-X` có thể bắt đầu một thực thể mới.
- `O` có thể xuất hiện ở bất kỳ vị trí nào.

Ví dụ:
```text
B-ORG I-ORG O B-PER   -> hợp lệ
O I-ORG O             -> không hợp lệ
B-ORG I-PER O         -> không hợp lệ
```

In [96]:
def validate_bio(labels):
    prev_prefix, prev_type = None, None
    for i, label in enumerate(labels):
        prefix, entity_type = split_bio_label(label)

        if prefix not in ("O", "B", "I"):
            return False, f"Label không hợp lệ: {label}"

        if prefix == "I" and (prev_prefix not in ("B", "I") or prev_type != entity_type):
            return False, f"I-{entity_type} tại vị trí {i} không thể đứng sau {prev_prefix}-{prev_type}"

        prev_prefix, prev_type = prefix, entity_type

    return True, None

ý tưởng validate_bio là tạo một hàm check các prefix và entity type có hợp lệ ko

đầu tiên alf gán 2 valiabe none

sau đó là chạy for trên tập labels

trong vòng lặp là kiểm tra từng phần như sau:

1. check xem có phải prefix là thuộc trong tập {"O", "B", "I"} hay ko nếu ko thì return tuple có dạng (False, message)

2. check xem I có đúng vị trí hay ko với 2 điều kiện gộp thành 1 đầu tiên là prefix có thuộc tập {"B", "I"} hay ko nếu ko thì return hoặc entity type có giống nhau ko nếu ko thì return

sau đó là gán giá trị 

cuối cùng là nếu hàm hứng ko có False nào thì return tuple có dạng là (True, None)

### Kiểm thử bắt buộc

In [97]:
# Bỏ comment sau khi hoàn thiện validate_bio
assert validate_bio(["B-ORG", "I-ORG", "O", "B-PER"])[0] is True
assert validate_bio(["O", "I-ORG", "O"])[0] is False
assert validate_bio(["B-ORG", "I-PER", "O"])[0] is False
assert validate_bio(["B-PER", "I-PER", "I-PER"])[0] is True

## 2.3. TODO 2 — Chuyển BIO labels → entity spans

Biểu diễn một span:
```python
{"start": 0, "end": 2, "type": "ORG", "text": "Công_ty Sao_Bắc"}
```
`end` dùng quy ước exclusive.

In [98]:
def bio_to_spans(tokens, labels):
    spans = []
    start, entity_type = None, None

    def close_span(end):
        return {
            "start": start,
            "end": end,
            "type": entity_type,
            "text": " ".join(tokens[start:end]),
        }

    for i, label in enumerate(labels):
        prefix, current_type = split_bio_label(label)

        if prefix == "B":
            if start is not None:
                spans.append(close_span(i))
            start, entity_type = i, current_type

        elif prefix == "O":
            if start is not None:
                spans.append(close_span(i))
            start, entity_type = None, None

    if start is not None:
        spans.append(close_span(len(tokens)))

    return spans

ý tưởng tạo hàm đầu tiên là 1 list chứa value, thứ 2 là biến 1 là dùng để lưu đoạn bắt đầu và 2 là lưu kiểu entity
sau đó tạo hêm hàm lưu vào list để đở phải bưng từng lần add vào cho từng điều kiện để thực hiện logic

tiếp tục tạo một vòng lập cho chạy từ prefix B và thêm vào list bằng hàm trên với điều kiện là start ko được none và gán start, entity_type bằng index hiện tại và curruent_type

nếu ko phải B thì check xem có phải là O hay ko nếu phải thì check xem start có none hay ko néu ko phải thì thêm vào list và gán giá trị start và entity type là none, none để kết biết đoạn kết thúc

sau vòng lặp thì check xem start có none ko nếu ko phải none thì thêm vào list với end là index của lần xuất hiện I cuối cùng trong đoạn khi gặp O hoặc ko gặp (là list label ko có O mà chỉ có B và I)

cuối cùng là return spans

In [99]:
tokens = ["Công_ty", "Sao_Bắc", "đang", "tuyển", "dụng", "Nguyễn", "An"]
labels = ["B-ORG", "I-ORG", "O", "O", "O", "B-PER", "I-PER"]
print(bio_to_spans(tokens, labels))

[{'start': 0, 'end': 2, 'type': 'ORG', 'text': 'Công_ty Sao_Bắc'}, {'start': 5, 'end': 7, 'type': 'PER', 'text': 'Nguyễn An'}]


## 2.4. TODO 3 — Chuyển entity spans → BIO labels

In [100]:
def spans_to_bio(tokens, spans):
    labels = ["O"] * len(tokens)
    for span in spans:
        start = span["start"]
        end = span["end"]
        entity_type = span["type"]

        labels[start] = f"B-{entity_type}"
        for i in range(start + 1, end):
            labels[i] = f"I-{entity_type}"
    return labels

In [101]:
tokens = ["Công_ty", "Sao_Bắc", "đang", "tuyển", "Nguyễn", "An"]
spans = [
    {"start": 0, "end": 2, "type": "ORG", "text": "Công_ty Sao_Bắc"},
    {"start": 4, "end": 6, "type": "PER", "text": "Nguyễn An"}
]
print(spans_to_bio(tokens, spans))

['B-ORG', 'I-ORG', 'O', 'O', 'B-PER', 'I-PER']


## 2.5. Round-trip test

In [102]:
tokens, labels = train_data[0]
spans = bio_to_spans(tokens, labels)
labels_reconstructed = spans_to_bio(tokens, spans)
print(spans)
assert labels == labels_reconstructed

[{'start': 0, 'end': 2, 'type': 'ORG', 'text': 'Công_ty Sao_Bắc'}, {'start': 4, 'end': 5, 'type': 'PER', 'text': 'Linh'}, {'start': 6, 'end': 7, 'type': 'POSITION', 'text': 'giám_đốc'}]


# 3. Bài 2 — Gazetteer-based NER

Pipeline:
```text
Raw text → Normalize → Match gazetteer → Resolve overlap → Entity candidates
```

In [103]:
gazetteer = {
    "ORG": {"fpt", "viettel", "vinfast", "apple", "công ty sao bắc", "công ty abc", "đại học quốc gia hà nội", "công ty minh long"},
    "LOC": {"hà nội", "hải phòng", "đà nẵng", "tp.hcm", "singapore"},
    "PER": {"linh", "mai", "nam", "lan", "nguyễn văn an"}
}

## 3.1. Normalize mẫu

In [104]:
def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

print(normalize_text("  Công ty   Sao Bắc  "))

công ty sao bắc


## 3.2. TODO 4 — Gazetteer matching

Tìm tất cả entity candidates và trả về character offsets, ví dụ:
```python
[{"start": 0, "end": 16, "text": "Công ty Sao Bắc", "type": "ORG"}]
```

In [105]:
def gazetteer_match(text, gazetteer):
    lower_text = text.lower()
    candidates = []

    for entity_type, entries in gazetteer.items():
        for entry in entries:
            entry_norm = normalize_text(entry)
            if not entry_norm:
                continue
            pattern = r"(?<!\w)" + re.escape(entry_norm) + r"(?!\w)"
            for m in re.finditer(pattern, lower_text):
                start, end = m.start(), m.end()
                candidates.append({
                    "start": start,
                    "end": end,
                    "text": text[start:end],
                    "type": entity_type,
                })
    return candidates

ý tưởng hàm
đầu tiên là tạo một variable lower text và một list chứa result

sau đó là tạo vongf for theo tham số là cac entity_type và entries trong tập gazetteer 

sau đó chạy vòng for tham số là entry trong trường entries và thực hiện việc normalize text cho entry

sau đó check xem entry norm có nội dung ko nếu ko thì continue 

tới tạo pattern với value là lấy text của entry_norm với điều kiện là chỉ lấy chính nó ví dụ như nin thì chỉ lấy nin ko lấy ninh hay ninh_dep_zai và hàm re.escape check xem có ký tự đặt biệt như là ".", "(", ")",... ko nếu có thì pattern là rổng

sau đó chạy vòng for với tham số là só từ được thìm khớp với pattern đã được lower text, trong vòng lặp tạo 2 variable là start và end với index ở start và index ở end sau đó add các giá trị vào candiates với các dict như sau: start, end, text (bắt đầu và kết thúc của nó), và type của nó

sau cuối cùng là trả về candidates

## 3.3. TODO 5 — Resolve overlap

Baseline: **Longest match wins**.

In [106]:
def spans_overlap(a, b):
    return not (a["end"] <= b["start"] or b["end"] <= a["start"])


def resolve_overlap_longest(candidates):
    sorted_cands = sorted(
        candidates,
        key=lambda s: (-(s["end"] - s["start"]), s["start"])
    )

    kept = []
    for cand in sorted_cands:
        if any(spans_overlap(cand, k) for k in kept):
            continue
        kept.append(cand)

    kept.sort(key=lambda s: s["start"])
    return kept

hàm đầu tiên là nếu a kết thúc trước khi b kịp bắt đầu hay b kết thúc trước khi a kịp bắt đầu thì return về false, và ngược lại return true. Dùng để kiểm tra xem 2 span có đè lên nhau (chồng lấn vị trí trong câu) hay không.

resolve_overlap_longest là bắt đầu tạo một variable sorted_cands, truyền vào là candidates, với key là lambda s, với s có dạng tuple là giá trị âm của (end trừ start, tức là độ dài span), và index start.

Sau đó tạo một list rỗng (kept).

Sau đó tạo một vòng for theo sorted_cands (đây chính là kết quả của phép sorted() vừa tạo ở dòng ngay phía trên, không phải biến lạ), rồi check xem bất kỳ cái spans_overlap(cand, k) chạy theo kept mà xuất hiện (trả True) thì dừng lại (bỏ qua candidate này), còn không thì tiếp tục và thực hiện add vào list kept. Sau đó sắp xếp kept từ bé đến lớn theo key của start, và cuối cùng là trả về kept list.

Dùng để giải quyết xung đột khi các span (kết quả từ gazetteer_match) chồng lấn vị trí nhau giữ lại span dài nhất, loại bỏ span ngắn hơn bị nó đè lên (chiến lược "longest match wins").

## 3.4. Thử nghiệm xuyên suốt

Dùng câu:
```text
Đại học Quốc gia Hà Nội mở trung tâm mới tại Hà Nội.
```

Giải thích: candidate nào overlap, candidate nào bị loại, và kết quả cuối có hợp lý không.

In [107]:
sample_text = "Đại học Quốc gia Hà Nội mở trung tâm mới tại Hà Nội."
candidates = gazetteer_match(sample_text, gazetteer)
pprint(candidates)
pprint(resolve_overlap_longest(candidates))

[{'end': 23, 'start': 0, 'text': 'Đại học Quốc gia Hà Nội', 'type': 'ORG'},
 {'end': 23, 'start': 17, 'text': 'Hà Nội', 'type': 'LOC'},
 {'end': 51, 'start': 45, 'text': 'Hà Nội', 'type': 'LOC'}]
[{'end': 23, 'start': 0, 'text': 'Đại học Quốc gia Hà Nội', 'type': 'ORG'},
 {'end': 51, 'start': 45, 'text': 'Hà Nội', 'type': 'LOC'}]


candidate Hà Nội bị overlap, nó bị lập 2 lần ở đoạn "Đại học Quốc gia Hà Nội"

candidate sau khi bị loại là start: 17, end 23 là "Hà Nội", kết quả cuối đã ko còn dòng đó nữa nên hợp lý

## 3.5. Câu hỏi phân tích

1. Vì sao gazetteer có precision cao nhưng recall có thể thấp?
2. Vì sao normalize quá mạnh có thể tạo false positive?
3. Khi nào `Longest match wins` có thể sai?
4. Nếu bài toán hỗ trợ **nested NER**, ta có nên luôn loại span nhỏ hơn không?

- Câu 1: Gazetteer chỉ match được entity có trong từ điển — entity mới, viết tắt, biến thể chính tả, tên riêng chưa cập nhật thì bị bỏ sót → recall thấp. Nhưng precision cao vì mỗi match là bằng chứng mạnh, ít false positive nếu word-boundary xử lý tốt.
- Câu 2: Normalize (bỏ dấu, lowercase, gộp space, bỏ ký tự đặc biệt) có thể làm các thực thể khác nhau bị trùng nhau — vd "Hà Nội" và "Hà nội", hoặc "FPT" và "fpt" match nhầm; tệ hơn là từ thông thường trùng tên entity.
- Câu 3: Khi có nested NER — span dài chứa thực thể ngắn mà thực thể ngắn mới đúng. VD: "Đại học Quốc gia Hà Nội" là ORG, nhưng "Hà Nội" bên trong cũng là LOC — nếu cần cả hai thì longest-wins sẽ loại mất LOC. Ngoài ra nếu gazetteer có entry dài nhưng sai, nó sẽ "nuốt" luôn entity đúng ngắn hơn.
- Câu 4: Không. Với nested NER cần giữ cả span cha lẫn span con, không dùng longest-wins. Nên:
* GGiữ tất cả candidate không trùng hoàn toàn, hoặc
* Dùng mô hình phân loại để quyết định span nào hợp lệ, hoặc
* Dùng BIO nhiều tầng nhãn (multi-layer BIO).

# 4. Bài 3 — Feature-based NER và Linear-chain CRF

Các nhóm feature:
- bề mặt: chữ hoa/thường, chữ số, độ dài, shape;
- ngữ cảnh: token trước/sau, cửa sổ ±2;
- gazetteer;
- đặc trưng tiếng Việt/dữ liệu.

## 4.1. Feature extraction mẫu

In [108]:
def token_features(sent, i):
    token = sent[i]
    low = token.lower()
    feats = {
        "bias": 1.0,
        "token.lower": low,
        "is_upper": token.isupper(),
        "is_title": token.istitle(),
        "has_digit": any(ch.isdigit() for ch in token),
        "length": len(token),
    }
    if i > 0:
        feats["prev.lower"] = sent[i - 1].lower()
    else:
        feats["BOS"] = True
    if i < len(sent) - 1:
        feats["next.lower"] = sent[i + 1].lower()
    else:
        feats["EOS"] = True
    return feats

print(token_features(train_data[0][0], 4))

{'bias': 1.0, 'token.lower': 'linh', 'is_upper': False, 'is_title': True, 'has_digit': False, 'length': 4, 'prev.lower': 'bà', 'next.lower': 'làm'}


## 4.2. TODO 6 — Mở rộng feature

Bổ sung ít nhất **5 feature** mới, bắt buộc có:
- một feature từ gazetteer;
- một feature về shape;
- một feature ngữ cảnh ±2;
- một feature đặc thù tiếng Việt/dữ liệu;
- một feature tự đề xuất.

In [109]:
def token_shape(token):
    """Tạo shape token: thay chữ hoa->X, chữ thường->x, số->d, giữ dấu câu."""
    shape = []
    for ch in token:
        if ch.isupper():
            shape.append("X")
        elif ch.islower():
            shape.append("x")
        elif ch.isdigit():
            shape.append("d")
        else:
            shape.append(ch)
    compressed = []
    for ch in shape:
        if not compressed or compressed[-1] != ch:
            compressed.append(ch)
    return "".join(compressed)


def token_features_improved(sent, i, gazetteer):
    """
    Feature cho token i, mở rộng so với baseline.
    """
    token = sent[i]
    low = token.lower()
    feats = {
        "bias": 1.0,
        "token.lower": low,
        "is_upper": token.isupper(),
        "is_title": token.istitle(),
        "has_digit": any(ch.isdigit() for ch in token),
        "length": len(token),
        # ===== Feature mới =====
        "shape": token_shape(token),
        "has_underscore": "_" in token, 
        "has_viet_diacritic": any(
            ch in "ăâáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵđ"
            for ch in low
        ),
    }

    for etype, entries in gazetteer.items():
        if any(low in e for e in entries):
            feats[f"gaz_{etype}"] = True
        else:
            feats[f"gaz_{etype}"] = False

    if i > 0:
        feats["prev.lower"] = sent[i - 1].lower()
        feats["prev.shape"] = token_shape(sent[i - 1])
    else:
        feats["BOS"] = True

    if i < len(sent) - 1:
        feats["next.lower"] = sent[i + 1].lower()
        feats["next.shape"] = token_shape(sent[i + 1])
    else:
        feats["EOS"] = True

    if i > 1:
        feats["prev2.lower"] = sent[i - 2].lower()
    else:
        feats["BOS2"] = True
    if i < len(sent) - 2:
        feats["next2.lower"] = sent[i + 2].lower()
    else:
        feats["EOS2"] = True

    feats["prev_is_punct"] = i > 0 and not sent[i - 1][0].isalnum()
    feats["next_is_punct"] = i < len(sent) - 1 and not sent[i + 1][0].isalnum()

    return feats

## 4.3. Chuẩn bị dữ liệu CRF

In [110]:
def sent2features(tokens, feature_fn):
    return [feature_fn(tokens, i) for i in range(len(tokens))]

X_train_basic = [sent2features(tokens, token_features) for tokens, labels in train_data]
y_train = [labels for tokens, labels in train_data]
X_test_basic = [sent2features(tokens, token_features) for tokens, labels in test_data]
y_test = [labels for tokens, labels in test_data]

## 4.4. Huấn luyện CRF baseline

CRF mô hình hóa trực tiếp:
\[
P(Y|X)
\]
và tìm:
\[
\hat{Y}=\arg\max_Y P(Y|X)
\]

In [111]:
try:
    import sklearn_crfsuite
    crf = sklearn_crfsuite.CRF(
        algorithm="lbfgs", c1=0.1, c2=0.1,
        max_iterations=100,
        all_possible_transitions=True
    )
    crf.fit(X_train_basic, y_train)
    y_pred_basic = crf.predict(X_test_basic)
    print("CRF baseline trained.")
    for (tokens, gold), pred in zip(test_data, y_pred_basic):
        print("\nTOKENS:", tokens)
        print("GOLD  :", gold)
        print("PRED  :", pred)
except ImportError:
    print("Chưa có sklearn-crfsuite. Hãy chạy cell cài đặt ở phần 0.")

CRF baseline trained.

TOKENS: ['Công_ty', 'Minh_Long', 'bổ_nhiệm', 'ông', 'Nam', 'làm', 'phó_giám_đốc', '.']
GOLD  : ['B-ORG', 'I-ORG', 'O', 'O', 'B-PER', 'O', 'B-POSITION', 'O']
PRED  : ['B-ORG', 'I-ORG', 'O', 'O', 'O', 'O', 'B-POSITION', 'O']

TOKENS: ['Viettel', 'mở', 'chi_nhánh', 'tại', 'Đà_Nẵng', '.']
GOLD  : ['B-ORG', 'O', 'O', 'O', 'B-LOC', 'O']
PRED  : ['B-ORG', 'O', 'O', 'O', 'B-LOC', 'O']

TOKENS: ['Lan', 'gia_nhập', 'FPT', 'vào', '01/10/2026', '.']
GOLD  : ['B-PER', 'O', 'B-ORG', 'O', 'B-DATE', 'O']
PRED  : ['B-ORG', 'O', 'O', 'O', 'B-DATE', 'O']


## 4.5. TODO 7 — CRF với feature cải tiến

1. Dùng `token_features_improved`.
2. Huấn luyện lại CRF.
3. So sánh với baseline.
4. Nêu ít nhất 3 feature hữu ích nhất theo quan sát của nhóm.

In [112]:
X_train_improved = [sent2features(tokens, lambda s, i: token_features_improved(s, i, gazetteer))
                    for tokens, labels in train_data]
X_test_improved  = [sent2features(tokens, lambda s, i: token_features_improved(s, i, gazetteer))
                    for tokens, labels in test_data]

crf_improved = sklearn_crfsuite.CRF(
    algorithm="lbfgs", c1=0.1, c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
crf_improved.fit(X_train_improved, y_train)
y_pred_improved = crf_improved.predict(X_test_improved)

print("CRF improved trained.")
for (tokens, gold), pred in zip(test_data, y_pred_improved):
    print("\nTOKENS:", tokens)
    print("GOLD  :", gold)
    print("PRED  :", pred)

CRF improved trained.

TOKENS: ['Công_ty', 'Minh_Long', 'bổ_nhiệm', 'ông', 'Nam', 'làm', 'phó_giám_đốc', '.']
GOLD  : ['B-ORG', 'I-ORG', 'O', 'O', 'B-PER', 'O', 'B-POSITION', 'O']
PRED  : ['B-ORG', 'I-ORG', 'O', 'O', 'B-PER', 'O', 'B-LOC', 'O']

TOKENS: ['Viettel', 'mở', 'chi_nhánh', 'tại', 'Đà_Nẵng', '.']
GOLD  : ['B-ORG', 'O', 'O', 'O', 'B-LOC', 'O']
PRED  : ['B-ORG', 'O', 'O', 'O', 'B-LOC', 'O']

TOKENS: ['Lan', 'gia_nhập', 'FPT', 'vào', '01/10/2026', '.']
GOLD  : ['B-PER', 'O', 'B-ORG', 'O', 'B-DATE', 'O']
PRED  : ['B-ORG', 'O', 'O', 'O', 'B-DATE', 'O']


# 5. Bài 4 — Đánh giá NER

### Token-level accuracy
\[
Accuracy = \frac{\text{số token dự đoán đúng nhãn}}{\text{tổng số token}}
\]

### Exact span matching
Một entity chỉ đúng khi `start`, `end`, `type` đều đúng.

### Strict vs Partial
- **Strict:** span + type khớp hoàn toàn.
- **Partial:** cho phép chồng lấp theo một quy ước định trước.

## 5.1. TODO 8 — Token accuracy

In [113]:
def token_accuracy(y_true, y_pred):
    correct = 0
    total = 0
    for true_labels, pred_labels in zip(y_true, y_pred):
        for t, p in zip(true_labels, pred_labels):
            total += 1
            if t == p:
                correct += 1
    return correct / total if total > 0 else 0.0

## 5.2. TODO 9 — Entity-level exact Precision/Recall/F1

\[
Precision = \frac{TP}{TP+FP},\quad
Recall = \frac{TP}{TP+FN},\quad
F1 = \frac{2PR}{P+R}
\]

In [114]:
def entity_prf(tokens_list, y_true, y_pred):
    tp = fp = fn = 0
    for tokens, true_labels, pred_labels in zip(tokens_list, y_true, y_pred):
        gold_spans = bio_to_spans(tokens, true_labels)
        pred_spans = bio_to_spans(tokens, pred_labels)

        gold_set = {(s["start"], s["end"], s["type"]) for s in gold_spans}
        pred_set = {(s["start"], s["end"], s["type"]) for s in pred_spans}

        tp += len(gold_set & pred_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return tp, fp, fn, precision, recall, f1

## 5.3. TODO 10 — Partial matching

Gợi ý dùng IoU:
\[
IoU = \frac{|Gold \cap Pred|}{|Gold \cup Pred|}
\]
Match nếu cùng type và `IoU >= 0.5`.

In [115]:
def span_iou(a, b):
    """Tính Intersection over Union giữa 2 span (dựa trên vị trí start/end)."""
    inter = max(0, min(a["end"], b["end"]) - max(a["start"], b["start"]))
    union = max(a["end"], b["end"]) - min(a["start"], b["start"])
    return inter / union if union > 0 else 0.0

In [116]:
def partial_match_prf(tokens_list, y_true, y_pred, iou_threshold=0.5):
    tp = fp = fn = 0
    for tokens, true_labels, pred_labels in zip(tokens_list, y_true, y_pred):
        gold_spans = bio_to_spans(tokens, true_labels)
        pred_spans = bio_to_spans(tokens, pred_labels)

        matched_gold = set()
        matched_pred = set()

        for score, gi, g in enumerate(gold_spans):
            for pi, p in enumerate(pred_spans):
                if pi in matched_pred:
                    continue
                if g["type"] == p["type"] and score >= iou_threshold:
                    matched_gold.add(gi)
                    matched_pred.add(pi)
                    break

        tp += len(matched_gold)
        fp += len(pred_spans) - len(matched_pred)
        fn += len(gold_spans) - len(matched_gold)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return tp, fp, fn, precision, recall, f1

In [117]:
tokens_list = [tokens for tokens, labels in test_data]
y_true = [labels for tokens, labels in test_data]

print("Token accuracy:", token_accuracy(y_true, y_pred_improved))
print("Entity exact PRF:", entity_prf(tokens_list, y_true, y_pred_improved))
print("Partial match PRF:", partial_match_prf(tokens_list, y_true, y_pred_improved))

Token accuracy: 0.85
Entity exact PRF: (5, 2, 3, 0.7142857142857143, 0.625, 0.6666666666666666)


ValueError: not enough values to unpack (expected 3, got 2)

## 5.4. Error analysis

Chọn ít nhất **5 lỗi** và phân loại: boundary, sai type, bỏ sót, false positive, unseen word, thiếu ngữ cảnh, tokenization, gazetteer.

| Câu | Gold | Prediction | Loại lỗi | Nguyên nhân | Cách cải tiến |
|---|---|---|---|---|---|
| Lan gia_nhập FPT vào 01/10/2026 . | FPT=B-ORG | FPT=O | Bỏ sót, thiếu ngữ cảnh | Trong train_data, "FPT" chỉ từng xuất hiện ở đầu câu (feature BOS=True); ở đây nó nằm giữa câu sau "gia_nhập" → model chưa học được ngữ cảnh này | Thêm câu train có FPT ở vị trí giữa câu; thêm feature prev.lower="gia_nhập" mạnh hơn (feature theo trigger từ) |
| Công_ty Minh_Long bổ_nhiệm ông Nam làm phó_giám_đốc . | phó_giám_đốc=B-POSITION | =B-LOC | Sai type, unseen word | "phó_giám_đốc" chưa từng xuất hiện trong train_data (chỉ có "giám_đốc") → model đoán nhầm loại dựa trên feature khác | Thêm nhiều biến thể chức danh vào train (phó_giám_đốc, trưởng_phòng...); thêm gazetteer riêng cho POSITION |
| Bà Hương gia_nhập Grab vào 05/11/2026 . | Grab=B-ORG | =O | Bỏ sót, unseen word, gazetteer | "Grab" không nằm trong gazetteer["ORG"] và chưa từng xuất hiện trong train → cả 2 nguồn thông tin (gazetteer + train) đều "mù" với từ này | Bổ sung "grab" vào gazetteer["ORG"]; mở rộng train_data với nhiều tên công ty hơn |
| Ông Tuấn làm_việc tại Công_ty ABC . | Ông=O, Tuấn=B-PER | Ông=B-ORG, Tuấn=O | Tokenization issue | Toàn bộ train_data viết tên người dạng 1 token nối _ (Nguyễn_Văn_An), nhưng câu này tách "Ông" và "Tuấn" thành 2 token riêng — model chưa từng học pattern "1 từ đơn + tên riêng đứng sau" | Thống nhất convention tokenize (luôn nối _ cho cụm tên); hoặc thêm feature nhận diện danh xưng ("ông", "bà", "anh"...) |
| Đại_học Bách_Khoa TP.HCM tổ_chức hội_thảo AI . | TP.HCM=I-ORG | =O | 	Boundary, gazetteer mâu thuẫn | "TP.HCM" có trong gazetteer["LOC"] nên feature gaz_LOC=True được bật, kéo model nghiêng về kết thúc span ORG sớm, dù đúng ra nó là phần tiếp theo của tên trường | Xử lý xung đột gazetteer đa nghĩa (1 từ có thể vừa là LOC vừa là phần của tên ORG); thêm feature "đang trong 1 span ORG chưa đóng" |
| Grab khai_trương văn_phòng mới tại Cần_Thơ . | khai_trương=O | =I-ORG | False positive, boundary | Từ "khai_trương" đứng ngay sau 1 ORG (Grab) nên model "lỡ" coi nó là phần tiếp theo của thực thể ORG | Thêm nhiều câu train có cấu trúc ORG + động từ để model học ranh giới rõ hơn (ORG dừng lại trước động từ) |


In [118]:
extra_sentences = [
    (["Bà", "Hương", "gia_nhập", "Grab", "vào", "05/11/2026", "."],
     ["O","B-PER","O","B-ORG","O","B-DATE","O"]),
    (["Ông", "Tuấn", "làm_việc", "tại", "Công_ty", "ABC", "."],
     ["O","B-PER","O","O","B-ORG","I-ORG","O"]),
    (["Đại_học", "Bách_Khoa", "TP.HCM", "tổ_chức", "hội_thảo", "AI", "."],
     ["B-ORG","I-ORG","I-ORG","O","O","O","O"]),
    (["Grab", "khai_trương", "văn_phòng", "mới", "tại", "Cần_Thơ", "."],
     ["B-ORG","O","O","O","O","B-LOC","O"]),
]

X_extra = [sent2features(tokens, lambda s,i: token_features_improved(s,i,gazetteer))
           for tokens, gold in extra_sentences]
y_extra_pred = crf_improved.predict(X_extra)

for (tokens, gold), pred in zip(extra_sentences, y_extra_pred):
    print("TOKENS:", tokens)
    print("GOLD  :", gold)
    print("PRED  :", pred)
    print()

TOKENS: ['Bà', 'Hương', 'gia_nhập', 'Grab', 'vào', '05/11/2026', '.']
GOLD  : ['O', 'B-PER', 'O', 'B-ORG', 'O', 'B-DATE', 'O']
PRED  : ['O' 'B-PER' 'O' 'O' 'O' 'B-DATE' 'O']

TOKENS: ['Ông', 'Tuấn', 'làm_việc', 'tại', 'Công_ty', 'ABC', '.']
GOLD  : ['O', 'B-PER', 'O', 'O', 'B-ORG', 'I-ORG', 'O']
PRED  : ['B-ORG' 'O' 'O' 'O' 'B-ORG' 'I-ORG' 'O']

TOKENS: ['Đại_học', 'Bách_Khoa', 'TP.HCM', 'tổ_chức', 'hội_thảo', 'AI', '.']
GOLD  : ['B-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'O', 'O']
PRED  : ['B-ORG' 'I-ORG' 'O' 'O' 'O' 'B-LOC' 'O']

TOKENS: ['Grab', 'khai_trương', 'văn_phòng', 'mới', 'tại', 'Cần_Thơ', '.']
GOLD  : ['B-ORG', 'O', 'O', 'O', 'O', 'B-LOC', 'O']
PRED  : ['B-ORG' 'I-ORG' 'O' 'O' 'O' 'B-LOC' 'O']



# 6. Bài 5 — Relation Extraction

Một relation thường được biểu diễn:
\[
(Entity_1, RelationType, Entity_2)
\]

Ví dụ:
```text
(Nguyễn Văn An, WORK_FOR, FPT)
```

In [119]:
relation_examples = [
    "Nguyễn Văn An làm việc tại FPT.",
    "Lan gia nhập Viettel.",
    "VinFast đặt nhà máy tại Hải Phòng.",
    "Apple mở văn phòng tại Singapore."
]

## 6.1. TODO 11 — Trích xuất relation bằng rule

Xây dựng ít nhất 2 loại relation:
- `WORK_FOR(Person, Organization)`
- `LOCATED_IN(Organization, Location)`

In [120]:
WORK_FOR_TRIGGERS = ["làm việc tại", "gia nhập", "làm tại"]
LOCATED_IN_TRIGGERS = ["đặt nhà máy tại", "đặt trụ sở tại", "mở văn phòng tại", "mở chi nhánh tại", "đặt tại"]

def extract_relations(text, gazetteer):
    entities = resolve_overlap_longest(gazetteer_match(text, gazetteer))
    entities = sorted(entities, key=lambda e: e["start"])
    lower_text = text.lower()
    relations = []

    for i, e1 in enumerate(entities):
        for e2 in entities[i+1:]:
            between = lower_text[e1["end"]:e2["start"]]

            if e1["type"] == "PER" and e2["type"] == "ORG":
                if any(trig in between for trig in WORK_FOR_TRIGGERS):
                    relations.append((e1["text"], "WORK_FOR", e2["text"]))

            if e1["type"] == "ORG" and e2["type"] == "LOC":
                if any(trig in between for trig in LOCATED_IN_TRIGGERS) or " tại " in (" "+between+" "):
                    relations.append((e1["text"], "LOCATED_IN", e2["text"]))

    return relations

## 6.2. Đánh giá relation

Tự tạo 5 câu gold, chạy hệ thống và tính Precision/Recall/F1.
Một relation chỉ đúng nếu entity 1, entity 2 và relation type đều đúng; với quan hệ có hướng, thứ tự entity cũng quan trọng.

In [121]:
# 6.2
gold_relations = {
    "Nguyễn Văn An làm việc tại FPT.": [("Nguyễn Văn An", "WORK_FOR", "FPT")],
    "Lan gia nhập Viettel.": [("Lan", "WORK_FOR", "Viettel")],
    "VinFast đặt nhà máy tại Hải Phòng.": [("VinFast", "LOCATED_IN", "Hải Phòng")],
    "Apple mở văn phòng tại Singapore.": [("Apple", "LOCATED_IN", "Singapore")],
    "FPT đặt trụ sở tại Hà Nội.": [("FPT", "LOCATED_IN", "Hà Nội")],
}

def evaluate_relations(gold_dict, gazetteer):
    tp = fp = fn = 0
    for text, gold_rels in gold_dict.items():
        pred_rels = extract_relations(text, gazetteer)
        gold_set, pred_set = set(gold_rels), set(pred_rels)
        tp += len(gold_set & pred_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)
    precision = tp/(tp+fp) if (tp+fp) else 0
    recall = tp/(tp+fn) if (tp+fn) else 0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) else 0
    return precision, recall, f1

p, r, f1 = evaluate_relations(gold_relations, gazetteer)
print(f"Precision: {p:.2f} | Recall: {r:.2f} | F1: {f1:.2f}")

Precision: 1.00 | Recall: 1.00 | F1: 1.00


# 7. Bài 6 — Event Extraction và Template Filling

Với event, đơn vị dự đoán thường gồm:
```text
Event Type + Trigger + Arguments và roles
```

Ví dụ:
```text
Công ty Sao Bắc bổ nhiệm bà Linh làm giám đốc vào ngày 12/09/2026.
```

Template:
```text
Event type: APPOINTMENT
Trigger: bổ nhiệm
Organization = Công ty Sao Bắc
Person = Linh
Position = giám đốc
Date = 12/09/2026
```

In [122]:
event_texts = [
    "Công ty Sao Bắc bổ nhiệm bà Linh làm giám đốc vào ngày 12/09/2026.",
    "Công ty Minh Long bổ nhiệm ông Nam làm phó giám đốc vào ngày 01/10/2026."
]

## 7.1. TODO 12 — Event trigger detection

In [123]:
def detect_event_trigger(text):
    trigger_map = {
        "APPOINTMENT": ["bổ nhiệm"],
        "RECRUITMENT": ["tuyển", "tuyển dụng"],
        "OPENING": ["mở", "khai trương"],
        "ACQUISITION": ["mua lại", "thâu tóm"],
    }
    for event_type, triggers in trigger_map.items():
        for trigger in triggers:
            match = re.search(r"(?<!\w)" + re.escape(trigger) + r"(?!\w)", text, re.IGNORECASE)
            if match:
                return {"event_type": event_type, "trigger": text[match.start():match.end()],
                        "start": match.start(), "end": match.end()}
    return None

for text in event_texts + ["VinFast mở nhà máy tại Hải Phòng."]:
    print(text, "->", detect_event_trigger(text))


Công ty Sao Bắc bổ nhiệm bà Linh làm giám đốc vào ngày 12/09/2026. -> {'event_type': 'APPOINTMENT', 'trigger': 'bổ nhiệm', 'start': 16, 'end': 24}
Công ty Minh Long bổ nhiệm ông Nam làm phó giám đốc vào ngày 01/10/2026. -> {'event_type': 'APPOINTMENT', 'trigger': 'bổ nhiệm', 'start': 18, 'end': 26}
VinFast mở nhà máy tại Hải Phòng. -> {'event_type': 'OPENING', 'trigger': 'mở', 'start': 8, 'end': 10}


Mở rộng ít nhất một event type khác, ví dụ `ACQUISITION`, `RECRUITMENT` hoặc `OPENING`.

## 7.2. TODO 13 — Slot filling

In [124]:
def fill_appointment_template(text, gazetteer):
    """Điền template APPOINTMENT bằng trigger, gazetteer và regex."""
    trigger = detect_event_trigger(text)
    if not trigger or trigger["event_type"] != "APPOINTMENT":
        return None

    entities = resolve_overlap_longest(gazetteer_match(text, gazetteer))
    organization = next((e["text"] for e in entities if e["type"] == "ORG"), None)
    person = next((e["text"] for e in entities if e["type"] == "PER"), None)
    date_match = re.search(r"\b\d{1,2}/\d{1,2}/\d{4}\b", text)
    position_match = re.search(
        r"\blàm\s+(.+?)(?=\s+vào\s+ngày|\s+ngày|[.,]|$)", text, re.IGNORECASE
    )
    return {
        "event_type": "APPOINTMENT",
        "trigger": trigger["trigger"],
        "arguments": {
            "Organization": organization,
            "Person": person,
            "Position": position_match.group(1).strip() if position_match else None,
            "Date": date_match.group(0) if date_match else None,
        }
    }


## 7.3. Template Filling

- **Slot filling:** điền từng trường riêng lẻ.
- **Template filling:** kết hợp nhiều slot thành một record cấu trúc.

Chạy trên toàn bộ `event_texts` và xuất JSON.

In [125]:
results = [fill_appointment_template(x, gazetteer) for x in event_texts]
print(json.dumps(results, ensure_ascii=False, indent=2))

[
  {
    "event_type": "APPOINTMENT",
    "trigger": "bổ nhiệm",
    "arguments": {
      "Organization": "Công ty Sao Bắc",
      "Person": "Linh",
      "Position": "giám đốc",
      "Date": "12/09/2026"
    }
  },
  {
    "event_type": "APPOINTMENT",
    "trigger": "bổ nhiệm",
    "arguments": {
      "Organization": "Công ty Minh Long",
      "Person": "Nam",
      "Position": "phó giám đốc",
      "Date": "01/10/2026"
    }
  }
]


# 8. Bài nâng cao — Subword alignment cho Transformer NER

Ví dụ minh họa:
```text
VinFast -> Vin, ##Fast
```

Nếu word-level label là `B-ORG`:

**First-subword-only**
```text
Vin -> B-ORG
##Fast -> -100
```

**Label-all-subwords**
```text
Vin -> B-ORG
##Fast -> I-ORG
```

Special tokens như `[CLS]`, `[SEP]`, `[PAD]` thường gán `-100` để bỏ qua khi tính loss.

## 8.1. TODO 14 — Align labels với `word_ids`

Với:
```python
word_ids = [None, 0, 0, 1, 2, None]
labels = ["B-ORG", "O", "B-LOC"]
```

Yêu cầu:
- `None -> -100`
- subword đầu lấy label gốc
- subword tiếp theo: `-100` hoặc đổi `B-X -> I-X` tùy chiến lược.

In [126]:
def align_labels_with_word_ids(word_ids, word_labels, first_subword_only=True):
    aligned = []
    previous_word_id = None
    for word_id in word_ids:
        if word_id is None:
            aligned.append(-100)
        elif not (0 <= word_id < len(word_labels)):
            raise IndexError(f"word_id ngoài phạm vi: {word_id}")
        elif word_id != previous_word_id:
            aligned.append(word_labels[word_id])
        elif first_subword_only:
            aligned.append(-100)
        else:
            label = word_labels[word_id]
            aligned.append("I-" + label[2:] if label.startswith("B-") else label)
        previous_word_id = word_id
    return aligned

word_ids = [None, 0, 0, 1, 2, None]
word_labels = ["B-ORG", "O", "B-LOC"]
print("First-subword-only:", align_labels_with_word_ids(word_ids, word_labels, True))
print("Label-all-subwords:", align_labels_with_word_ids(word_ids, word_labels, False))


First-subword-only: [-100, 'B-ORG', -100, 'O', 'B-LOC', -100]
Label-all-subwords: [-100, 'B-ORG', 'I-ORG', 'O', 'B-LOC', -100]


# 9. Bài bonus — So sánh HMM, CRF, BiLSTM-CRF, Transformer

| Mô hình | Input representation | Context | Quan hệ giữa nhãn | Feature thủ công | Ưu điểm | Hạn chế |
|---|---|---|---|---|---|---|
| HMM | | | | | | |
| CRF | | | | | | |
| BiLSTM-CRF | | | | | | |
| Transformer | | | | | | |

Trả lời:
1. Vì sao HMM là mô hình sinh còn CRF là mô hình phân biệt?
2. Vì sao BiLSTM giúp giảm nhu cầu feature engineering?
3. Vì sao CRF vẫn có thể hữu ích phía trên BiLSTM?
4. Transformer token classification có nhất thiết phải dùng CRF không?

# 10. Báo cáo kết quả cuối notebook

### 10.1. Kết quả
- Token accuracy:
- Exact entity Precision:
- Exact entity Recall:
- Exact entity F1:
- Partial entity F1:
- Relation F1:

### 10.2. Ba lỗi phổ biến nhất
1.
2.
3.

### 10.3. Ba cải tiến quan trọng nhất
1.
2.
3.

### 10.4. Nhận xét
Trong khoảng **150–250 từ**, nhận xét sự khác nhau giữa Gazetteer-based NER, Feature-based CRF và Neural/Transformer NER; giải thích khi nào nên dùng hệ thống hybrid.

# 11. Checklist trước khi nộp

- [ ] BIO validator chạy đúng.
- [ ] BIO ↔ spans chạy round-trip.
- [ ] Gazetteer matcher hoạt động.
- [ ] Resolve overlap hoạt động.
- [ ] Có ít nhất 5 feature mới cho CRF.
- [ ] Có baseline CRF và improved CRF.
- [ ] Có token accuracy.
- [ ] Có exact entity P/R/F1.
- [ ] Có partial matching.
- [ ] Có error analysis ít nhất 5 trường hợp.
- [ ] Có ít nhất 2 loại relation.
- [ ] Có event trigger + slot/template filling.
- [ ] Có phần nhận xét cuối.